# Google Stock Price Prediction using LSTM
4-layer stacked LSTM with Dropout, EarlyStopping, rolling forecast, and uncertainty estimation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Load data
dataset_train = pd.read_csv('Google_Stock_Price_Train.csv')
dataset_test  = pd.read_csv('Google_Stock_Price_Test.csv')

# Fix comma-formatted numbers
for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    dataset_train[col] = dataset_train[col].astype(str).str.replace(',', '').astype(float)
    dataset_test[col]  = dataset_test[col].astype(str).str.replace(',', '').astype(float)

print('Train shape:', dataset_train.shape)
print('Test shape: ', dataset_test.shape)
dataset_train.head()

In [ ]:
# Use Close price (industry standard)
training_set = dataset_train[['Close']].values
sc = MinMaxScaler(feature_range=(0, 1))
training_set_scaled = sc.fit_transform(training_set)

X_train, y_train = [], []
for i in range(60, len(training_set_scaled)):
    X_train.append(training_set_scaled[i-60:i, 0])
    y_train.append(training_set_scaled[i, 0])

X_train = np.array(X_train)
y_train = np.array(y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
print('X_train shape:', X_train.shape)

In [ ]:
regressor = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    LSTM(units=50, return_sequences=True),
    Dropout(0.2),
    LSTM(units=50, return_sequences=True),
    Dropout(0.2),
    LSTM(units=50, return_sequences=True),
    Dropout(0.2),
    LSTM(units=50),
    Dropout(0.2),
    Dense(units=1)
])

regressor.compile(optimizer='adam', loss='mean_squared_error')
regressor.summary()

In [ ]:
import os
os.makedirs('images', exist_ok=True)  # ← add this line

# Train with EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    min_delta=0.0001,
    restore_best_weights=True
)

history = regressor.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig('images/training_loss.png', bbox_inches='tight')
plt.show()

In [ ]:
real_stock_price = dataset_test[['Close']].values

dataset_total = pd.concat((dataset_train['Close'], dataset_test['Close']), axis=0)
inputs = dataset_total[len(dataset_total) - len(dataset_test) - 60:].values
inputs = inputs.reshape(-1, 1)
inputs = sc.transform(inputs)

X_test = []
for i in range(60, len(inputs)):
    X_test.append(inputs[i-60:i, 0])

X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

predicted_stock_price = regressor.predict(X_test)
predicted_stock_price = sc.inverse_transform(predicted_stock_price)

In [ ]:
# Evaluation metrics
import os
os.makedirs('images', exist_ok=True)

mse = mean_squared_error(real_stock_price, predicted_stock_price)
mae = mean_absolute_error(real_stock_price, predicted_stock_price)
r2  = r2_score(real_stock_price, predicted_stock_price)

print(f'MSE: {mse:.4f}')
print(f'MAE: {mae:.4f}')
print(f'R²:  {r2:.4f}')

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(12, 5))
plt.plot(real_stock_price, color='red', label='Real Google Stock Price')
plt.plot(predicted_stock_price, color='blue', label='Predicted Google Stock Price')
plt.title('Google Stock Price Prediction')
plt.xlabel('Time')
plt.ylabel('Stock Price (USD)')
plt.legend()
plt.savefig('images/actual_vs_predicted.png', bbox_inches='tight')
plt.show()

In [ ]:
# Rolling forecast - predict 30 days into future
def rolling_forecast(model, initial_input, steps):
    forecasts = []
    current_input = initial_input.copy()
    for _ in range(steps):
        prediction = model.predict(current_input, verbose=0)
        forecasts.append(prediction[0, 0])
        current_input = np.roll(current_input, -1, axis=1)
        current_input[0, -1, 0] = prediction[0, 0]  # fixed: use scalar
    return np.array(forecasts)

future_scaled = rolling_forecast(regressor, X_test[-1:], 30)
future_prices = sc.inverse_transform(future_scaled.reshape(-1, 1))

plt.figure(figsize=(10, 4))
plt.plot(future_prices, color='green', marker='o', markersize=3)
plt.title('30-Day Future Price Forecast')
plt.xlabel('Days Ahead')
plt.ylabel('Predicted Price (USD)')
plt.savefig('images/future_forecast.png', bbox_inches='tight')
plt.show()

In [ ]:
# Uncertainty estimation using Monte Carlo Dropout
def predict_with_uncertainty(model, X, n_iter=100):
    predictions = np.zeros((n_iter, X.shape[0], 1))  # fixed shape
    for i in range(n_iter):
        predictions[i] = model(X, training=True)  # Dropout active
    return predictions.mean(axis=0), predictions.std(axis=0)

mean_pred, std_pred = predict_with_uncertainty(regressor, X_test)
mean_pred = sc.inverse_transform(mean_pred)
std_pred  = std_pred * (sc.data_max_ - sc.data_min_)  # scale std back

plt.figure(figsize=(12, 5))
plt.plot(real_stock_price, color='red', label='Real Price')
plt.plot(mean_pred, color='blue', label='Mean Prediction')
plt.fill_between(
    range(len(mean_pred)),
    (mean_pred - 2*std_pred).flatten(),
    (mean_pred + 2*std_pred).flatten(),
    color='gray', alpha=0.3, label='95% Confidence Interval'
)
plt.title('Prediction with Uncertainty Bounds')
plt.xlabel('Time')
plt.ylabel('Stock Price (USD)')
plt.legend()
plt.savefig('images/uncertainty_bounds.png', bbox_inches='tight')
plt.show()